# 🏌️ GolfCaddy-1B Training Notebook

This notebook fine-tunes TinyLlama on golf-specific Q&A data.

**Runtime**: GPU (T4 recommended)
**Time**: ~2-4 hours
**Output**: Custom golf LLM model (~300MB)

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate trl anthropic

# Check GPU
!nvidia-smi

## Step 2: Generate Training Data

In [ ]:
# Set your Anthropic API key
import os
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

# Upload golf_dataset_generator.py to Colab
# Then run:
!python golf_dataset_generator.py --output golf_data.json --count 2000

## Step 3: Fine-Tune Model

In [ ]:
# Upload finetune_golf_llm.py to Colab
# Then run training:
!python finetune_golf_llm.py \
    --data golf_data_training.json \
    --output golfcaddy-1b \
    --epochs 3 \
    --batch-size 4 \
    --test \
    --export

## Step 4: Convert to GGUF (for iOS deployment)

In [ ]:
# Clone llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make

# Convert to GGUF
!python llama.cpp/convert.py golfcaddy-1b/merged --outtype f16 --outfile golfcaddy-1b-f16.gguf

# Quantize to Q4_K_M (optimal size/quality)
!./llama.cpp/quantize golfcaddy-1b-f16.gguf golfcaddy-1b-Q4_K_M.gguf Q4_K_M

## Step 5: Test Quantized Model

In [ ]:
# Test the GGUF model
!./llama.cpp/main \
    -m golfcaddy-1b-Q4_K_M.gguf \
    -p "You are CaddieChat Pro. Q: Where should I place the ball if it lands on cart path? A:" \
    -n 200 \
    --temp 0.7

## Step 6: Download Model

In [ ]:
# Check file size
!ls -lh golfcaddy-1b-Q4_K_M.gguf

# Download to your computer
from google.colab import files
files.download('golfcaddy-1b-Q4_K_M.gguf')

print("✅ Model ready for iOS deployment!")